In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint
from datasets import load_dataset, Dataset
import os
from pathlib import Path
import torch

In [22]:
os.environ["HF_CACHE"] = "/sadata/nikkothari/cache/hf/hub/"

OUTPUT_DIR = "/sadata/nikkothari/kai-scheduler-lab/train/ckpts/"
MODEL_NAME = "google/gemma-4-E2B-it"
TRAIN_DATA_PATH = "/sadata/nikkothari/kai-scheduler-lab/train/data/train.json"
EVAL_DATA_PATH = "/sadata/nikkothari/kai-scheduler-lab/train/data/eval.json"
MAX_LENGTH = 4096

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████| 1951/1951 [00:00<00:00, 7751.60it/s]


In [9]:
model.device

device(type='cpu')

In [10]:
model

Gemma4ForConditionalGeneration(
  (model): Gemma4Model(
    (vision_tower): Gemma4VisionModel(
      (patch_embedder): Gemma4VisionPatchEmbedder(
        (input_proj): Linear(in_features=768, out_features=768, bias=False)
      )
      (encoder): Gemma4VisionEncoder(
        (rotary_emb): Gemma4VisionRotaryEmbedding()
        (layers): ModuleList(
          (0-15): 16 x Gemma4VisionEncoderLayer(
            (self_attn): Gemma4VisionAttention(
              (q_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (k_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (v_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (o_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=Fals

In [30]:
train_dataset = load_dataset("trl-lib/Capybara", split="train")
test_dataset = load_dataset("trl-lib/Capybara", split="test")
train_dataset[0], test_dataset[0]

({'source': 'GPT4LLM',
  'messages': [{'content': 'Recommend a movie to watch.\n', 'role': 'user'},
   {'content': 'I would recommend the movie, "The Shawshank Redemption" which is a classic drama film starring Tim Robbins and Morgan Freeman. This film tells a powerful story about hope and resilience, as it follows the story of a young man who is wrongfully convicted of murder and sent to prison. Amidst the harsh realities of prison life, the protagonist forms a bond with a fellow inmate, and together they navigate the challenges of incarceration, while holding on to the hope of eventual freedom. This timeless movie is a must-watch for its moving performances, uplifting message, and unforgettable storytelling.',
    'role': 'assistant'},
   {'content': 'Describe the character development of Tim Robbins\' character in "The Shawshank Redemption".',
    'role': 'user'},
   {'content': 'In "The Shawshank Redemption", Tim Robbins plays the character of Andy Dufresne, a banker who is wrongfu

In [99]:
from transformers import DataCollatorForLanguageModeling

collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
    pad_to_multiple_of=8,
    return_tensors="pt",
)

### First Way

In [104]:
collator(
    [{"input_ids": [32, 31, 32], "attention_mask": [1,1,1]},
     {"input_ids": [32, 31, 32, 2,2,2,2,2,2,2,2,2,2], "attention_mask": [1,1,1,1,1,1,1,1,1,1,1,1,1]}]
)

{'input_ids': tensor([[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0, 32, 31, 32],
        [ 0,  0,  0, 32, 31, 32,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2]]), 'attention_mask': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1],
        [0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]), 'labels': tensor([[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100,   32,   31,   32],
        [-100, -100, -100,   32,   31,   32,    2,    2,    2,    2,    2,    2,
            2,    2,    2,    2]])}

### Second Way

In [101]:
messages = [
    [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a short joke about saving RAM."},
        {"role": "assistant", "content": "RAM.. RAM.."}
    ],
    [
        {"role": "user", "content": "Write a short joke."},
        {"role": "assistant", "content": "RANDOM.."}
    ]
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    return_dict=True,
    truncation=True,
    max_length=MAX_LENGTH,
    padding=False,                # Padding happens in the collator
    add_generation_prompt=False,  # These conversations include the answers
)

features = [
    {key: values[i] for key, values in inputs.items()}
    for i in range(len(inputs["input_ids"]))
]

batch = collator(features)

for key, value in batch.items():
    print(key, value.shape)
    print(value)

input_ids torch.Size([2, 40])
tensor([[     0,      0,      0,      0,      0,      0,      2,    105,   9731,
            107,   3048,    659,    496,  11045,  16326, 236761,    106,    107,
            105,   2364,    107,   6974,    496,   2822,  31481,   1003,  16011,
          24051, 236761,    106,    107,    105,   4368,    107,   4849,    856,
          24051,    856,    106,    107],
        [     0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      0,      0,      0,      0,      0,      0,
              0,      0,      0,      2,    105,   2364,    107,   6974,    496,
           2822,  31481, 236761,    106,    107,    105,   4368,    107, 135072,
           4206,    856,    106,    107]])
attention_mask torch.Size([2, 40])
tensor([[0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

### Final Way

In [ ]:
tokenize_function = lambda messages: tokenizer.apply_chat_template(
    messages["messages"],
    tokenize=True,
    return_dict=True,
    truncation=True,
    max_length=MAX_LENGTH,
    padding=False,
    add_generation_prompt=False,
)

remove_columns = list(train_dataset[0].keys())
tokenized_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=remove_columns)
tokenized_dataset = list(tokenized_dataset)

In [124]:
batch = collator(tokenized_dataset[:2])

In [126]:
for key, value in batch.items():
    print(key, value.shape)
    print(value)

input_ids torch.Size([2, 984])
tensor([[   0,    0,    0,  ...,  105, 4368,  107],
        [   0,    0,    0,  ...,  105, 4368,  107]])
attention_mask torch.Size([2, 984])
tensor([[0, 0, 0,  ..., 1, 1, 1],
        [0, 0, 0,  ..., 1, 1, 1]])
labels torch.Size([2, 984])
tensor([[-100, -100, -100,  ...,  105, 4368,  107],
        [-100, -100, -100,  ...,  105, 4368,  107]])


In [132]:
get_last_checkpoint("/sadata/nikkothari/kai-scheduler-lab/train/ckpts")

'/sadata/nikkothari/kai-scheduler-lab/train/ckpts/checkpoint-600'